In [2]:
# MCP SCENARIO: “Smart IT Helpdesk Assistant”
# 🧩 Scenario Background

# You are working in a company called ABC Corp.

# Employees face issues like:

# VPN not working
# Printer not responding
# Software errors

# 👉 Instead of calling IT support, employees use an AI Helpdesk Bot.

# 🤖 What this Bot Should Do

# When a user types a problem:

# Understand the issue
# Decide if a ticket is needed
# Identify:
# Category (Network / Hardware / General)
# Priority (High / Medium)
# Create a ticket
# Show confirmation
# 🧠 How MCP Fits Here
# Component	Role in Scenario
# Agent	Helpdesk Bot
# MCP Layer	Decision + Tool calling
# Tool	Ticket Creation System
# User	Employee




# ============================================
# STEP 0: DATABASE (Simulated storage)
# ============================================

tickets_db = []  # This stores all tickets


# ============================================
# STEP 1: TOOL (MCP TOOL)
# ============================================

def create_ticket(issue, priority, category):
    """
    This function simulates a TOOL in MCP
    In real world → API / Database / ServiceNow
    """

    ticket_id = f"INC{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)

    return ticket


# ============================================
# STEP 2: AGENT REASONING (LLM SIMULATION)
# ============================================

def analyze_input(user_input):
    """
    Simulates how an LLM understands user input
    Extracts:
    - category
    - priority
    """

    text = user_input.lower()

    # 🔹 Category Detection
    if "vpn" in text:
        category = "network"
    elif "printer" in text:
        category = "hardware"
    elif "email" in text:
        category = "software"
    else:
        category = "general"

    # 🔹 Priority Detection
    if "urgent" in text or "immediately" in text:
        priority = "high"
    elif "slow" in text:
        priority = "low"
    else:
        priority = "medium"

    return category, priority


# ============================================
# STEP 3: DECISION ENGINE (MCP CORE)
# ============================================

def should_call_tool(user_input):
    """
    Decides whether to call a tool or not
    This is MCP decision layer
    """

    keywords = ["issue", "problem", "ticket", "not working"]

    return any(word in user_input.lower() for word in keywords)


# ============================================
# STEP 4: MCP ORCHESTRATOR
# ============================================

def mcp_agent(user_input):
    """
    This is the MAIN MCP FLOW
    It connects:
    Agent → Decision → Tool → Response
    """

    print("\n🧠 Agent received input:", user_input)

    # STEP 4.1: Decision
    if should_call_tool(user_input):

        print("➡️ Decision: Tool call required")

        # STEP 4.2: Analyze input
        category, priority = analyze_input(user_input)

        print(f"📊 Extracted → Category: {category}, Priority: {priority}")

        # STEP 4.3: Prepare payload (MCP format)
        payload = {
            "issue": user_input,
            "priority": priority,
            "category": category
        }

        print("📦 MCP Payload:", payload)

        # STEP 4.4: Call tool
        result = create_ticket(**payload)

        print("⚙️ Tool executed successfully")

        # STEP 4.5: Final response
        return f"""
        ✅ Ticket Created Successfully!

        Ticket ID: {result['ticket_id']}
        Issue: {result['issue']}
        Category: {result['category']}
        Priority: {result['priority']}
        """

    else:
        print("➡️ Decision: No tool needed (AI response)")

        return "🤖 AI Response: Please describe your issue clearly."


# ============================================
# STEP 5: RUN INTERACTIVE LOOP
# ============================================

print("🚀 MCP Demo Started (Type 'exit' to stop)\n")

while True:

    user_input = input("Enter your query: ")

    if user_input.lower() == "exit":
        print("👋 Exiting MCP demo...")
        break

    response = mcp_agent(user_input)
    print(response)



🚀 MCP Demo Started (Type 'exit' to stop)

Enter your query: vpn issue

🧠 Agent received input: vpn issue
➡️ Decision: Tool call required
📊 Extracted → Category: network, Priority: medium
📦 MCP Payload: {'issue': 'vpn issue', 'priority': 'medium', 'category': 'network'}
⚙️ Tool executed successfully

        ✅ Ticket Created Successfully!

        Ticket ID: INC1000
        Issue: vpn issue
        Category: network
        Priority: medium
        
Enter your query: chatgpt issue 

🧠 Agent received input: chatgpt issue 
➡️ Decision: Tool call required
📊 Extracted → Category: general, Priority: medium
📦 MCP Payload: {'issue': 'chatgpt issue ', 'priority': 'medium', 'category': 'general'}
⚙️ Tool executed successfully

        ✅ Ticket Created Successfully!

        Ticket ID: INC1001
        Issue: chatgpt issue 
        Category: general
        Priority: medium
        
Enter your query: laptop issue

🧠 Agent received input: laptop issue
➡️ Decision: Tool call required
📊 Extracted → C

In [3]:
!pip install groq
import os
from groq import Groq
from google.colab import userdata

# Load API key securely
api_key = userdata.get("GROQ_API_KEY")

client = Groq(api_key=api_key)

# ============================================
# STEP 0: DATABASE
# ============================================

tickets_db = []

# ============================================
# STEP 1: TOOL
# ============================================

def create_ticket(issue, priority, category):
    ticket_id = f"INC{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)
    return ticket


# ============================================
# STEP 2: LLM ANALYSIS (REPLACES RULES)
# ============================================

def analyze_with_llm(user_input):
    """
    LLM decides:
    - should_create_ticket
    - category
    - priority
    """

    prompt = f"""
You are an IT helpdesk assistant.

Analyze the user issue and respond in JSON format:

{{
  "create_ticket": true/false,
  "category": "network/hardware/software/general",
  "priority": "high/medium/low"
}}

User Input: "{user_input}"
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",  # fast + powerful
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    output = response.choices[0].message.content

    try:
        import json
        parsed = json.loads(output)
    except:
        parsed = {
            "create_ticket": True,
            "category": "general",
            "priority": "medium"
        }

    return parsed


# ============================================
# STEP 3: MCP AGENT
# ============================================

def mcp_agent(user_input):

    print("\n🧠 Agent received:", user_input)

    # LLM Decision
    decision = analyze_with_llm(user_input)

    print("🤖 LLM Decision:", decision)

    if decision["create_ticket"]:

        payload = {
            "issue": user_input,
            "priority": decision["priority"],
            "category": decision["category"]
        }

        print("📦 MCP Payload:", payload)

        result = create_ticket(**payload)

        return f"""
✅ Ticket Created Successfully!

Ticket ID: {result['ticket_id']}
Issue: {result['issue']}
Category: {result['category']}
Priority: {result['priority']}
"""

    else:
        return "🤖 AI Response: No ticket required. Try basic troubleshooting."


# ============================================
# STEP 4: RUN LOOP
# ============================================

print("🚀 LLM MCP Helpdesk Started (type 'exit')\n")

while True:

    user_input = input("Enter issue: ")

    if user_input.lower() == "exit":
        print("👋 Exiting...")
        break

    response = mcp_agent(user_input)
    print(response)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 3.7 MB/s eta 0:00:00
🚀 LLM MCP Helpdesk Started (type 'exit')

Enter issue: Help me with Python code

🧠 Agent received: Help me with Python code
🤖 LLM Decision: {'create_ticket': True, 'category': 'general', 'priority': 'medium'}
📦 MCP Payload: {'issue': 'Help me with Python code', 'priority': 'medium', 'category': 'general'}

✅ Ticket Created Successfully!

Ticket ID: INC1000
Issue: Help me with Python code
Category: general
Priority: medium

Enter issue: hello

🧠 Agent received: hello
🤖 LLM Decision: {'create_ticket': False, 'category': 'general', 'priority': 'low'}
🤖 AI Response: No ticket required. Try basic troubleshooting.
Enter issue: exit
👋 Exiting...


In [4]:
# MCP SCENARIO: “Smart HR Onboarding Assistant”

# 🧩 Scenario Background
# You are working in a company called XYZ Corp.
# New employees often face challenges during onboarding, such as:
# - Trouble accessing payroll portal
# - Confusion about leave policies
# - Difficulty setting up email accounts
# - Questions about training schedules

# 👉 Instead of emailing HR or waiting for responses, employees use an AI Onboarding Bot.

# 🤖 What this Bot Should Do
# When a new hire types a question/problem:
# - Understand the query (e.g., “I can’t log into payroll”)
# - Decide if escalation to HR is needed
# - Identify:
# - Category (Payroll / Policy / IT Setup / Training)
# - Priority (High / Medium)
# - Create a support ticket if required
# - Provide instant guidance (FAQs, step-by-step instructions)
# - Show confirmation and next steps

# 🧠 How MCP Fits Here
# |  |  |
# |  |  |
# |  |  |
# |  |  |
# |  |  |

# This way, the MCP framework is reused in a Human Resources context, where the AI assistant streamlines onboarding, reduces HR workload, and ensures employees feel supported from day one.

# Would you like me to design another variation in a customer service setting (like retail or banking), so you can compare how MCP adapts across industries?

# ============================================
# INSTALL & IMPORTS
# ============================================

!pip install groq

import os
import json
import re
from datetime import datetime
from groq import Groq
from google.colab import userdata

# Load API key
api_key = userdata.get("GROQ_API_KEY")
client = Groq(api_key=api_key)

# ============================================
# STEP 0: DATABASE (Simulated)
# ============================================

tickets_db = []

# ============================================
# STEP 1: TOOL (Ticket Creation)
# ============================================

def create_ticket(issue, category, priority):
    ticket_id = f"HR{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "category": category,
        "priority": priority,
        "status": "Open",
        "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    tickets_db.append(ticket)
    return ticket

# ============================================
# STEP 2: SAFE JSON EXTRACTION
# ============================================

def extract_json(text):
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except:
            return None
    return None

# ============================================
# STEP 3: LLM ANALYSIS (MCP BRAIN)
# ============================================

def analyze_with_llm(user_input):

    prompt = f"""
You are an HR onboarding assistant.

Analyze the employee query and return ONLY JSON:

{{
  "create_ticket": true/false,
  "category": "payroll/policy/it setup/training",
  "priority": "high/medium",
  "response": "helpful guidance for the user"
}}

User Query: "{user_input}"
"""

    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )

        output = response.choices[0].message.content
        parsed = extract_json(output)

    except Exception as e:
        print("❌ LLM Error:", e)
        parsed = None

    # Fallback (important)
    if not parsed:
        parsed = {
            "create_ticket": True,
            "category": "general",
            "priority": "medium",
            "response": "Please contact HR for assistance."
        }

    return parsed

# ============================================
# STEP 4: VALIDATION (Safety Layer)
# ============================================

def validate_decision(decision):

    valid_categories = ["payroll", "policy", "it setup", "training"]
    valid_priorities = ["high", "medium"]

    category = decision.get("category", "policy").lower()
    priority = decision.get("priority", "medium").lower()

    if category not in valid_categories:
        category = "policy"

    if priority not in valid_priorities:
        priority = "medium"

    return {
        "create_ticket": decision.get("create_ticket", True),
        "category": category,
        "priority": priority,
        "response": decision.get("response", "")
    }

# ============================================
# STEP 5: MCP AGENT
# ============================================

def mcp_hr_agent(user_input):

    print("\n🧠 Received Query:", user_input)

    # LLM Decision
    decision_raw = analyze_with_llm(user_input)
    decision = validate_decision(decision_raw)

    print("🤖 LLM Decision:", decision)

    # Always give guidance
    guidance = f"\n💡 Guidance: {decision['response']}\n"

    # Decide tool call
    if decision["create_ticket"]:

        payload = {
            "issue": user_input,
            "category": decision["category"],
            "priority": decision["priority"]
        }

        print("📦 MCP Payload:", payload)

        result = create_ticket(**payload)

        return f"""
{guidance}
✅ HR Ticket Created!

Ticket ID: {result['ticket_id']}
Category: {result['category']}
Priority: {result['priority']}
Status: {result['status']}

📌 HR Team will contact you soon.
"""

    else:
        return f"""
{guidance}
🤖 No ticket required. You're good to go!
"""

# ============================================
# STEP 6: RUN LOOP
# ============================================

print("🚀 HR Onboarding MCP Assistant Started (type 'exit')\n")

while True:

    user_input = input("Ask your onboarding question: ")

    if user_input.lower() == "exit":
        print("👋 Exiting...")
        break

    response = mcp_hr_agent(user_input)
    print(response)


🚀 HR Onboarding MCP Assistant Started (type 'exit')

Ask your onboarding question: I am not able to access payroll portal

🧠 Received Query: I am not able to access payroll portal
🤖 LLM Decision: {'create_ticket': True, 'category': 'payroll', 'priority': 'high', 'response': "Sorry to hear that you're having trouble accessing the payroll portal. Can you please try clearing your browser cache and cookies, then attempt to log in again? If the issue persists, our IT team will be happy to assist you in resolving the matter. Please provide your employee ID and the error message you're seeing, if any."}
📦 MCP Payload: {'issue': 'I am not able to access payroll portal', 'category': 'payroll', 'priority': 'high'}


💡 Guidance: Sorry to hear that you're having trouble accessing the payroll portal. Can you please try clearing your browser cache and cookies, then attempt to log in again? If the issue persists, our IT team will be happy to assist you in resolving the matter. Please provide your emp

In [5]:
# ============================================
# STEP 0: DATABASE (Simulated)
# ============================================

tickets_db = []

# ============================================
# STEP 1: TOOL (Ticket Creation)
# ============================================

def create_ticket(issue, category, priority):
    ticket_id = f"HR{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "category": category,
        "priority": priority,
        "status": "Open"
    }

    tickets_db.append(ticket)
    return ticket


# ============================================
# STEP 2: RULE-BASED ANALYSIS (NO LLM)
# ============================================

def analyze_input(user_input):
    text = user_input.lower()

    # 🔹 Category Detection
    if "payroll" in text or "salary" in text:
        category = "Payroll"
    elif "leave" in text or "policy" in text:
        category = "Policy"
    elif "email" in text or "login" in text or "account" in text:
        category = "IT Setup"
    elif "training" in text:
        category = "Training"
    else:
        category = "Policy"

    # 🔹 Priority Detection
    if "urgent" in text or "immediately" in text:
        priority = "High"
    else:
        priority = "Medium"

    return category, priority


# ============================================
# STEP 3: DECISION ENGINE (MCP CORE)
# ============================================

def should_create_ticket(user_input):
    text = user_input.lower()

    keywords = [
        "not working",
        "unable",
        "can't",
        "cannot",
        "issue",
        "problem",
        "error",
        "fail"
    ]

    return any(word in text for word in keywords)


# ============================================
# STEP 4: GUIDANCE SYSTEM (SMART RESPONSES)
# ============================================

def get_guidance(user_input):
    text = user_input.lower()

    if "payroll" in text:
        return "Try logging in again or reset your password. If issue persists, contact HR."

    elif "leave" in text or "policy" in text:
        return "You can check leave policy in the HR portal under 'Policies' section."

    elif "email" in text or "account" in text:
        return "Ensure your credentials are correct. Try resetting your password."

    elif "training" in text:
        return "Training schedule is shared via email. Please check your inbox."

    else:
        return "Please check HR portal or contact HR for more details."


# ============================================
# STEP 5: MCP AGENT
# ============================================

def mcp_hr_agent(user_input):

    print("\n🧠 Received Query:", user_input)

    # Step 1: Analyze input
    category, priority = analyze_input(user_input)

    print(f"📊 Category: {category}, Priority: {priority}")

    # Step 2: Get guidance
    guidance = get_guidance(user_input)

    # Step 3: Decision (MCP Layer)
    if should_create_ticket(user_input):

        print("➡️ Decision: Create Ticket")

        payload = {
            "issue": user_input,
            "category": category,
            "priority": priority
        }

        print("📦 MCP Payload:", payload)

        result = create_ticket(**payload)

        return f"""
💡 Guidance: {guidance}

✅ HR Ticket Created!

Ticket ID: {result['ticket_id']}
Category: {result['category']}
Priority: {result['priority']}
Status: {result['status']}

📌 HR Team will contact you soon.
"""

    else:
        print("➡️ Decision: No Ticket Needed")

        return f"""
💡 Guidance: {guidance}

🤖 No ticket required. You're good to go!
"""


# ============================================
# STEP 6: RUN LOOP
# ============================================

print("🚀 HR Onboarding Assistant Started (Rule-Based MCP)\n")

while True:

    user_input = input("Ask your onboarding question: ")

    if user_input.lower() == "exit":
        print("👋 Exiting...")
        break

    response = mcp_hr_agent(user_input)
    print(response)

🚀 HR Onboarding Assistant Started (Rule-Based MCP)

Ask your onboarding question: I can't access payroll What is leave policy? My email is not working When is training?

🧠 Received Query: I can't access payroll What is leave policy? My email is not working When is training?
📊 Category: Payroll, Priority: Medium
➡️ Decision: Create Ticket
📦 MCP Payload: {'issue': "I can't access payroll What is leave policy? My email is not working When is training?", 'category': 'Payroll', 'priority': 'Medium'}

💡 Guidance: Try logging in again or reset your password. If issue persists, contact HR.

✅ HR Ticket Created!

Ticket ID: HR1000
Category: Payroll
Priority: Medium
Status: Open

📌 HR Team will contact you soon.

Ask your onboarding question: When is training?

🧠 Received Query: When is training?
📊 Category: Training, Priority: Medium
➡️ Decision: No Ticket Needed

💡 Guidance: Training schedule is shared via email. Please check your inbox.

🤖 No ticket required. You're good to go!

Ask your onbo

In [6]:
# MCP SCENARIO: “Smart Banking Support Assistant”

# 🧩 Scenario Background
# You are working in a company called FinTrust Bank.
# Customers often face issues such as:
# - Credit card not working
# - Trouble with online banking login
# - Queries about loan status
# - Transaction disputes

# 👉 Instead of calling customer care, customers use an AI Banking Support Bot.

# 🤖 What this Bot Should Do
# When a customer types a problem:
# - Understand the issue (e.g., “My card was declined”)
# - Decide if escalation to a human agent is needed
# - Identify:
# - Category (Card Services / Online Banking / Loans / Transactions)
# - Priority (High / Medium)
# - Create a support ticket if required
# - Provide instant guidance (FAQs, troubleshooting steps, policy info)
# - Show confirmation and next steps

# 🧠 How MCP Fits Here
# |  |  |
# |  |  |
# |  |  |
# |  |  |
# |  |  |

# This way, MCP is applied in a financial services context, where the AI assistant reduces call center load, provides quick resolutions, and ensures customers feel supported with secure, reliable guidance.

# Would you like me to craft one more in a healthcare setting (like hospital patient support), so you can see how MCP adapts to critical service environments?


# ============================================
# STEP 0: DATABASE (Simulated)
# ============================================

tickets_db = []

# ============================================
# STEP 1: TOOL (Ticket Creation)
# ============================================

def create_ticket(issue, category, priority):
    ticket_id = f"BNK{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "category": category,
        "priority": priority,
        "status": "Open"
    }

    tickets_db.append(ticket)
    return ticket


# ============================================
# STEP 2: ANALYSIS (Rule-Based)
# ============================================

def analyze_input(user_input):
    text = user_input.lower()

    # 🔹 Category Detection
    if "card" in text:
        category = "Card Services"
    elif "login" in text or "password" in text:
        category = "Online Banking"
    elif "loan" in text:
        category = "Loans"
    elif "transaction" in text or "payment" in text:
        category = "Transactions"
    else:
        category = "Transactions"

    # 🔹 Priority Detection
    if "urgent" in text or "fraud" in text or "declined" in text:
        priority = "High"
    else:
        priority = "Medium"

    return category, priority


# ============================================
# STEP 3: DECISION ENGINE (MCP CORE)
# ============================================

def should_create_ticket(user_input):
    text = user_input.lower()

    keywords = [
        "not working",
        "failed",
        "declined",
        "error",
        "issue",
        "problem",
        "fraud",
        "unable"
    ]

    return any(word in text for word in keywords)


# ============================================
# STEP 4: GUIDANCE SYSTEM
# ============================================

def get_guidance(user_input):
    text = user_input.lower()

    if "card" in text:
        return "Check if your card is active and has sufficient balance. Try again or contact support."

    elif "login" in text or "password" in text:
        return "Use 'Forgot Password' option to reset your login credentials."

    elif "loan" in text:
        return "You can check your loan status in the banking app under the 'Loans' section."

    elif "transaction" in text or "payment" in text:
        return "If money is deducted but not received, wait 24 hours before raising a complaint."

    else:
        return "Please contact customer support for further assistance."


# ============================================
# STEP 5: MCP AGENT
# ============================================

def banking_mcp_agent(user_input):

    print("\n🧠 Customer Query:", user_input)

    # Step 1: Analyze input
    category, priority = analyze_input(user_input)
    print(f"📊 Category: {category}, Priority: {priority}")

    # Step 2: Get guidance
    guidance = get_guidance(user_input)

    # Step 3: Decision (MCP)
    if should_create_ticket(user_input):

        print("➡️ Decision: Create Ticket")

        payload = {
            "issue": user_input,
            "category": category,
            "priority": priority
        }

        print("📦 MCP Payload:", payload)

        result = create_ticket(**payload)

        return f"""
💡 Guidance: {guidance}

✅ Support Ticket Created!

Ticket ID: {result['ticket_id']}
Category: {result['category']}
Priority: {result['priority']}
Status: {result['status']}

📌 Our support team will contact you shortly.
"""

    else:
        print("➡️ Decision: No Ticket Needed")

        return f"""
💡 Guidance: {guidance}

🤖 No ticket required. Follow the above steps.
"""


# ============================================
# STEP 6: RUN LOOP
# ============================================

print("🏦 Banking Support MCP Assistant Started (type 'exit')\n")

while True:

    user_input = input("Enter your issue: ")

    if user_input.lower() == "exit":
        print("👋 Exiting...")
        break

    response = banking_mcp_agent(user_input)
    print(response)

🏦 Banking Support MCP Assistant Started (type 'exit')

Enter your issue: My card is not working

🧠 Customer Query: My card is not working
📊 Category: Card Services, Priority: Medium
➡️ Decision: Create Ticket
📦 MCP Payload: {'issue': 'My card is not working', 'category': 'Card Services', 'priority': 'Medium'}

💡 Guidance: Check if your card is active and has sufficient balance. Try again or contact support.

✅ Support Ticket Created!

Ticket ID: BNK1000
Category: Card Services
Priority: Medium
Status: Open

📌 Our support team will contact you shortly.

Enter your issue: I forgot my login password

🧠 Customer Query: I forgot my login password
📊 Category: Online Banking, Priority: Medium
➡️ Decision: No Ticket Needed

💡 Guidance: Use 'Forgot Password' option to reset your login credentials.

🤖 No ticket required. Follow the above steps.

Enter your issue: exit
👋 Exiting...


In [7]:
# ============================================
# INSTALL & IMPORTS
# ============================================

!pip install groq

import json
import re
from datetime import datetime
from groq import Groq
from google.colab import userdata

# Load API key
api_key = userdata.get("GROQ_API_KEY")
client = Groq(api_key=api_key)

# ============================================
# STEP 0: DATABASE
# ============================================

tickets_db = []

# ============================================
# STEP 1: TOOL
# ============================================

def create_ticket(issue, category, priority):
    ticket_id = f"BNK{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "category": category,
        "priority": priority,
        "status": "Open",
        "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    tickets_db.append(ticket)
    return ticket


# ============================================
# STEP 2: SAFE JSON EXTRACTION
# ============================================

def extract_json(text):
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except:
            return None
    return None


# ============================================
# STEP 3: LLM ANALYSIS (MCP BRAIN)
# ============================================

def analyze_with_llm(user_input):

    prompt = f"""
You are a banking support assistant.

Analyze the customer query and return ONLY JSON:

{{
  "create_ticket": true/false,
  "category": "card services/online banking/loans/transactions",
  "priority": "high/medium",
  "response": "helpful guidance"
}}

User Query: "{user_input}"
"""

    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )

        output = response.choices[0].message.content
        parsed = extract_json(output)

    except Exception as e:
        print("❌ LLM Error:", e)
        parsed = None

    # Fallback
    if not parsed:
        parsed = {
            "create_ticket": True,
            "category": "transactions",
            "priority": "medium",
            "response": "Please contact customer support."
        }

    return parsed


# ============================================
# STEP 4: VALIDATION (Safety Layer)
# ============================================

def validate_decision(decision):

    valid_categories = [
        "card services",
        "online banking",
        "loans",
        "transactions"
    ]

    valid_priorities = ["high", "medium"]

    category = decision.get("category", "transactions").lower()
    priority = decision.get("priority", "medium").lower()

    if category not in valid_categories:
        category = "transactions"

    if priority not in valid_priorities:
        priority = "medium"

    return {
        "create_ticket": decision.get("create_ticket", True),
        "category": category.title(),
        "priority": priority.title(),
        "response": decision.get("response", "")
    }


# ============================================
# STEP 5: MCP AGENT
# ============================================

def banking_mcp_agent(user_input):

    print("\n🧠 Customer Query:", user_input)

    # LLM Decision
    decision_raw = analyze_with_llm(user_input)
    decision = validate_decision(decision_raw)

    print("🤖 LLM Decision:", decision)

    guidance = f"\n💡 Guidance: {decision['response']}\n"

    # MCP Decision
    if decision["create_ticket"]:

        payload = {
            "issue": user_input,
            "category": decision["category"],
            "priority": decision["priority"]
        }

        print("📦 MCP Payload:", payload)

        result = create_ticket(**payload)

        return f"""
{guidance}
✅ Support Ticket Created!

Ticket ID: {result['ticket_id']}
Category: {result['category']}
Priority: {result['priority']}
Status: {result['status']}

📌 Our banking team will contact you soon.
"""

    else:
        return f"""
{guidance}
🤖 No ticket required. Issue resolved via guidance.
"""


# ============================================
# STEP 6: RUN LOOP
# ============================================

print("🏦 Banking MCP Assistant (LLM) Started (type 'exit')\n")

while True:

    user_input = input("Enter your issue: ")

    if user_input.lower() == "exit":
        print("👋 Exiting...")
        break

    response = banking_mcp_agent(user_input)
    print(response)

🏦 Banking MCP Assistant (LLM) Started (type 'exit')

Enter your issue: My credit card is not working

🧠 Customer Query: My credit card is not working
🤖 LLM Decision: {'create_ticket': True, 'category': 'Card Services', 'priority': 'High', 'response': "Sorry to hear that your credit card is not working. Can you please try checking if your card is expired or if you have sufficient funds? Additionally, ensure that your card is properly inserted or swiped, and that you are using the correct PIN. If the issue persists, we'll be happy to assist you further."}
📦 MCP Payload: {'issue': 'My credit card is not working', 'category': 'Card Services', 'priority': 'High'}


💡 Guidance: Sorry to hear that your credit card is not working. Can you please try checking if your card is expired or if you have sufficient funds? Additionally, ensure that your card is properly inserted or swiped, and that you are using the correct PIN. If the issue persists, we'll be happy to assist you further.

✅ Support Ti

In [8]:
# MCP SCENARIO: Weather Tool MCP Server

# 🧩 Objective
# Create a Weather Tool MCP Server that any AI agent can use to fetch real-time weather data
# and assist users in making decisions based on weather conditions.

# ================================
# 🧠 MCP Components
# ================================

# Model:
# - Understands user query like:
#   "What's the weather in Delhi?"
#   "Will it rain tomorrow in Mumbai?"

# Context:
# - Extracts:
#   - Location (e.g., Delhi, Mumbai)
#   - Date/Time (today, tomorrow)
# - Maintains conversation history if needed

# Protocol:
# - Defines interaction between agent and tool
# - Agent sends request → Tool processes → Returns structured weather data

# ================================
# 🛠️ Weather Tool Definition
# ================================

# def get_weather(location):
#     # Example API call (pseudo)
#     return {
#         "location": location,
#         "temperature": "28°C",
#         "condition": "Partly Cloudy",
#         "rain_chance": "20%"
#     }

# ================================
# 🤖 Agent + MCP Flow
# ================================

# User Query:
# "Will it rain today in Delhi?"

# Step 1: Model understands intent → Weather Query
# Step 2: Context extracts → location = Delhi, time = today
# Step 3: Protocol calls tool → get_weather("Delhi")
# Step 4: Tool returns data
# Step 5: Agent generates response

# ================================
# 🔄 End-to-End Flow
# ================================

# User → AI Agent
# → Intent Detection (Weather Query)
# → Extract Context (Location, Time)
# → Call Weather Tool (MCP)
# → Receive Weather Data
# → Generate Final Answer

# ================================
# 📌 Sample Use Case
# ================================

# User: "Should I carry an umbrella in Mumbai today?"

# Tool Output:
# {
#   "temperature": "30°C",
#   "condition": "Rainy",
#   "rain_chance": "80%"
# }

# Final Response:
# "Yes, you should carry an umbrella. There is a high chance of rain (80%) in Mumbai today."

# ================================
# 🎯 Benefits
# ================================

# ✔ Reusable tool for any AI agent
# ✔ Real-time decision making
# ✔ Modular MCP architecture
# ✔ Can integrate with travel, logistics, or planning systems

# ============================================
# STEP 0: DATABASE / MOCK DATA
# ============================================

weather_data = {
    "delhi": "32°C, Sunny",
    "mumbai": "28°C, Humid",
    "london": "15°C, Cloudy",
    "new york": "10°C, Rainy"
}

# ============================================
# STEP 1: TOOL (Weather API Simulation)
# ============================================

def get_weather(city):
    city = city.lower()
    return weather_data.get(city, "Weather data not available")


# ============================================
# STEP 2: ANALYSIS (Extract city)
# ============================================

def analyze_input(user_input):
    text = user_input.lower()

    for city in weather_data.keys():
        if city in text:
            return city

    return None


# ============================================
# STEP 3: DECISION ENGINE (MCP)
# ============================================

def should_call_tool(user_input):
    keywords = ["weather", "temperature", "climate"]
    return any(word in user_input.lower() for word in keywords)


# ============================================
# STEP 4: MCP AGENT
# ============================================

def mcp_weather_agent(user_input):

    print("\n🧠 User Query:", user_input)

    # Decision
    if should_call_tool(user_input):

        print("➡️ Decision: Call Weather Tool")

        # Extract city
        city = analyze_input(user_input)

        if not city:
            return "❌ Please specify a valid city."

        print(f"📍 Extracted City: {city}")

        # Tool Call
        result = get_weather(city)

        return f"""
🌦️ Weather Info:

City: {city.title()}
Condition: {result}
"""

    else:
        print("➡️ Decision: No Tool Needed")

        return "🤖 Ask me about weather like: 'Weather in Delhi'"


# ============================================
# STEP 5: RUN LOOP
# ============================================

print("🌦️ Weather MCP Server Started (type 'exit')\n")

while True:

    user_input = input("Enter your query: ")

    if user_input.lower() == "exit":
        print("👋 Exiting...")
        break

    response = mcp_weather_agent(user_input)
    print(response)

🌦️ Weather MCP Server Started (type 'exit')

Enter your query: Weather in Delhi

🧠 User Query: Weather in Delhi
➡️ Decision: Call Weather Tool
📍 Extracted City: delhi

🌦️ Weather Info:

City: Delhi
Condition: 32°C, Sunny

Enter your query: Weather in lucknow

🧠 User Query: Weather in lucknow
➡️ Decision: Call Weather Tool
❌ Please specify a valid city.
Enter your query: Weather in mumbai

🧠 User Query: Weather in mumbai
➡️ Decision: Call Weather Tool
📍 Extracted City: mumbai

🌦️ Weather Info:

City: Mumbai
Condition: 28°C, Humid

Enter your query: Weather in london

🧠 User Query: Weather in london
➡️ Decision: Call Weather Tool
📍 Extracted City: london

🌦️ Weather Info:

City: London
Condition: 15°C, Cloudy

Enter your query: exit
👋 Exiting...


In [10]:
# ============================================
# INSTALL
# ============================================

!pip install groq gradio

# ============================================
# IMPORTS
# ============================================

import json
import re
import gradio as gr
from groq import Groq
from google.colab import userdata

# Load API key
api_key = userdata.get("GROQ_API_KEY")
client = Groq(api_key=api_key)

# ============================================
# TOOL 1: WEATHER
# ============================================

weather_data = {
    "delhi": "32°C, Sunny",
    "mumbai": "28°C, Humid",
    "london": "15°C, Cloudy"
}

def get_weather(city):
    return weather_data.get(city.lower(), "Weather data not available")

# ============================================
# TOOL 2: NEWS
# ============================================

news_data = {
    "india": "India: Tech sector growing rapidly 🚀",
    "world": "Global markets show mixed trends 📊",
    "sports": "India wins thrilling cricket match 🏏"
}

def get_news(topic):
    return news_data.get(topic.lower(), "No news available")

# ============================================
# JSON EXTRACTOR
# ============================================

def extract_json(text):
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except:
            return None
    return None

# ============================================
# LLM DECISION (MCP BRAIN)
# ============================================

def analyze_with_llm(user_input):

    prompt = f"""
You are an AI assistant with tools.

Decide which tool to use.

Return ONLY JSON:

{{
  "tool": "weather/news/none",
  "input": "city or topic",
  "response": "normal reply if no tool"
}}

User Query: "{user_input}"
"""

    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )

        output = response.choices[0].message.content
        parsed = extract_json(output)

    except Exception as e:
        print("Error:", e)
        parsed = None

    if not parsed:
        parsed = {
            "tool": "none",
            "input": "",
            "response": "Sorry, I couldn't understand."
        }

    return parsed

# ============================================
# MCP AGENT
# ============================================

def mcp_multi_tool_agent(user_input):

    decision = analyze_with_llm(user_input)

    tool = decision.get("tool")
    tool_input = decision.get("input")

    # 🔹 WEATHER TOOL
    if tool == "weather":
        if not tool_input:
            return "❌ Please specify a city."
        result = get_weather(tool_input)
        return f"🌦️ Weather in {tool_input.title()}: {result}"

    # 🔹 NEWS TOOL
    elif tool == "news":
        if not tool_input:
            return "❌ Please specify a topic (india/world/sports)."
        result = get_news(tool_input)
        return f"📰 News ({tool_input.title()}): {result}"

    # 🔹 NO TOOL
    else:
        return decision.get("response")


# ============================================
# GRADIO UI
# ============================================

def chatbot_response(message, history):
    return mcp_multi_tool_agent(message)

chat_ui = gr.ChatInterface(
    fn=chatbot_response,
    title="🤖 Multi-Tool MCP Assistant",
    description="Ask weather or news (e.g., 'Weather in Delhi', 'News about India')"
)

chat_ui.launch()

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://54feef3ef02e18d6b1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
